In [12]:
"""
Standalone AQI hyperparameter tuning script.

Tuning objective:
- Train on 2020-2023 using a subset of SEEN stations.
- Validate on 2024 using separate held-out stations selected from the SEEN group.
- Select configurations by mean per-station MAE (macro MAE).
- The project's truly UNSEEN stations are not used for tuning at all.
- Writes best_hyperparameters.json for the separate final model script.

This preserves the final 2025 evaluation on the project's designated unseen
stations as a genuine holdout for spatial generalization.
"""

from pathlib import Path
import json
import numpy as np
import pandas as pd
import cupy as cp

from sklearn.model_selection import ParameterSampler, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


# ============================================================
# CONFIGURATION — keep in sync with the final model script
# ============================================================

INPUT_CSV = Path("../Datasets/Final_Dataset.csv")
OUTPUT_DIR = Path("../Outputs/XGBoost")

TARGETS = ["AQI_future_1", "AQI_future_24"]

WEATHER_COLUMNS = [
    "temperature_2m (°C)",
    "relative_humidity_2m (%)",
    "dew_point_2m (°C)",
    "precipitation (mm)",
    "surface_pressure (hPa)",
    "wind_speed_10m (km/h)",
    "wind_direction_10m (°)",
    "shortwave_radiation (W/m²)",
    "cloud_cover (%)",
]

RANDOM_SEED = 42
DEVICE = "cuda"  # Change to "cpu" if CUDA is unavailable.

# Fraction of the project's SEEN station IDs held out for tuning validation.
# These are only tuning-validation stations; the project's designated
# UNSEEN stations remain untouched until final 2025 evaluation.
TUNING_VALIDATION_FRACTION = 0.20

N_ITER = 30
MAX_ESTIMATORS = 1500
EARLY_STOPPING_ROUNDS = 75

PARAM_DISTRIBUTIONS = {
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.08],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.05, 0.1, 0.3, 1.0],
    "reg_alpha": [0, 0.001, 0.01, 0.1, 1.0],
    "reg_lambda": [1, 2, 5, 10, 20],
}

EXCLUDE = {
    "Day", "Month", "Year", "Time", "Day_of_Week", "Season",
    "Station_ID", "Station_Class", "Station_Split",
    "_Timestamp", "_year", "_target_time",
    *TARGETS,
}


# ============================================================
# LOAD AND PREPARE DATA (same feature logic as final script)
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Dataset not found: {INPUT_CSV.resolve()}")

df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns.")

required = {
    "Year", "Month", "Day", "Time", "Station_ID", "Station_Class",
    "latitude", "longitude", "elevation", *TARGETS, *WEATHER_COLUMNS,
}
missing = sorted(required - set(df.columns))
if missing:
    raise KeyError(f"Required columns missing from dataset: {missing}")

df = df.dropna(subset=["Station_ID", "Station_Class"]).copy()
df["Station_ID"] = df["Station_ID"].astype(str).str.strip()
df["Station_Class"] = (
    df["Station_Class"].astype(str).str.strip().str.lower()
)

unexpected = set(df["Station_Class"].unique()) - {"seen", "unseen"}
if unexpected:
    raise ValueError(
        "Station_Class must contain only 'seen'/'unseen'; "
        f"found {sorted(unexpected)}"
    )

date_text = (
    df["Year"].astype("Int64").astype(str) + "-"
    + df["Month"].astype("Int64").astype(str) + "-"
    + df["Day"].astype("Int64").astype(str) + " "
    + df["Time"].astype(str).str.strip()
)
df["_Timestamp"] = pd.to_datetime(
    date_text,
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)
df = df.dropna(subset=["_Timestamp"]).copy()
df["_year"] = df["_Timestamp"].dt.year

df["Hour"] = df["_Timestamp"].dt.hour

print("Hour values found:", sorted(df["Hour"].unique()))
print("Missing Hour values:", df["Hour"].isna().sum())

for col in ["latitude", "longitude", "elevation", *TARGETS]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[TARGETS] = df[TARGETS].replace([np.inf, -np.inf], np.nan)

df = pd.get_dummies(
    df, columns=["Day_of_Week", "Season"], dtype=np.uint8
)

class_counts = df.groupby("Station_ID")["Station_Class"].nunique()
if (class_counts > 1).any():
    raise ValueError(
        "Some Station_IDs have inconsistent Station_Class values: "
        f"{class_counts[class_counts > 1].index.tolist()[:20]}"
    )

seen_ids = sorted(set(df.loc[df["Station_Class"] == "seen", "Station_ID"]))
unseen_ids = set(df.loc[df["Station_Class"] == "unseen", "Station_ID"])

if len(seen_ids) < 5 or not unseen_ids:
    raise ValueError(
        f"Need enough seen stations and at least one unseen station; "
        f"seen={len(seen_ids)}, unseen={len(unseen_ids)}"
    )

# Reserve a deterministic subset of seen stations for spatial tuning validation.
tuning_train_station_ids, tuning_val_station_ids = train_test_split(
    seen_ids,
    test_size=TUNING_VALIDATION_FRACTION,
    random_state=RANDOM_SEED,
    shuffle=True,
)

tuning_train_station_ids = set(tuning_train_station_ids)
tuning_val_station_ids = set(tuning_val_station_ids)

# Safety checks: designated unseen stations must not appear in either group.
assert not (tuning_train_station_ids & unseen_ids)
assert not (tuning_val_station_ids & unseen_ids)
assert not (tuning_train_station_ids & tuning_val_station_ids)

print(f"Project seen stations: {len(seen_ids)}")
print(f"Project unseen stations held untouched: {len(unseen_ids)}")
print(f"Tuning training stations: {len(tuning_train_station_ids)}")
print(f"Tuning validation stations: {len(tuning_val_station_ids)}")

pd.DataFrame({
    "Station_ID": sorted(tuning_train_station_ids),
    "Tuning_Role": "train_station",
}).to_csv(OUTPUT_DIR / "tuning_train_stations.csv", index=False)

pd.DataFrame({
    "Station_ID": sorted(tuning_val_station_ids),
    "Tuning_Role": "validation_station",
}).to_csv(OUTPUT_DIR / "tuning_validation_stations.csv", index=False)

features = [
    col for col in df.select_dtypes(include=np.number).columns
    if col not in EXCLUDE
    and not col.startswith("AQI_future_")
    and not col.lower().startswith(("target", "label", "future_"))
]

missing_weather = [c for c in WEATHER_COLUMNS if c not in df.columns]
if missing_weather:
    raise KeyError(f"Weather columns missing after preprocessing: {missing_weather}")

model_features = {
    "A": [c for c in features if c not in WEATHER_COLUMNS],
    "B": features,
}

with open(
    OUTPUT_DIR / "tuning_feature_columns.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(model_features, f, indent=2)

if not model_features["A"] or not model_features["B"]:
    raise ValueError("A feature set is empty.")

# The same candidate configurations are tested for all four experiments.
candidate_params = list(ParameterSampler(
    PARAM_DISTRIBUTIONS,
    n_iter=N_ITER,
    random_state=RANDOM_SEED,
))


# ============================================================
# HELPERS
# ============================================================

def clean_X(frame, columns):
    return frame.loc[:, columns].replace([np.inf, -np.inf], np.nan)


def macro_station_mae(y_true, predictions, station_ids):
    scores = pd.DataFrame({
        "Station_ID": station_ids.to_numpy(),
        "Actual": np.asarray(y_true),
        "Predicted": np.asarray(predictions),
    })
    scores["Absolute_Error"] = (scores["Actual"] - scores["Predicted"]).abs()
    per_station_mae = scores.groupby("Station_ID")["Absolute_Error"].mean()
    return float(per_station_mae.mean()), int(per_station_mae.size)




def predict_gpu_batched(model, X, batch_size=65536, best_iteration=None):
    """
    Predict using CuPy-backed batches on the same GPU as the XGBoost model.
    Feature order must match the columns used for fitting.
    """
    booster = model.get_booster()
    predictions = []

    if best_iteration is None:
        iteration_range = (0, 0)  # all trees
    else:
        iteration_range = (0, int(best_iteration) + 1)

    for start in range(0, len(X), batch_size):
        stop = min(start + batch_size, len(X))

        # X is already in the model's feature order; cast each batch to float32.
        cpu_batch = X.iloc[start:stop].to_numpy(
            dtype=np.float32, copy=True
        )
        gpu_batch = cp.asarray(cpu_batch)

        batch_predictions = booster.inplace_predict(
            gpu_batch,
            iteration_range=iteration_range,
            validate_features=False,
        )
        predictions.append(cp.asnumpy(batch_predictions))

        del gpu_batch, cpu_batch, batch_predictions

    return np.concatenate(predictions) if predictions else np.empty(0)


def make_tuning_model(params, validation_station_ids):
    # Early stopping must optimize the same station-balanced objective used
    # to rank candidates, not ordinary row-weighted MAE.
    station_ids = np.asarray(validation_station_ids)

    def macro_mae_eval(y_true, y_pred):
        scores = pd.DataFrame({
            "Station_ID": station_ids,
            "Actual": np.asarray(y_true),
            "Predicted": np.asarray(y_pred),
        })
        scores["Absolute_Error"] = (
            scores["Actual"] - scores["Predicted"]
        ).abs()
        return float(
            scores.groupby("Station_ID")["Absolute_Error"].mean().mean()
        )

    return XGBRegressor(
        **params,
        n_estimators=MAX_ESTIMATORS,
        objective="reg:squarederror",
        eval_metric=macro_mae_eval,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method="hist",
        device=DEVICE,
    )


# ============================================================
# TUNE: TRAIN ON SOME SEEN STATIONS, VALIDATE ON OTHER SEEN STATIONS
# ============================================================

PARAMS = {target: {} for target in TARGETS}
all_results = []
summary_rows = []

for target in TARGETS:
    horizon = 1 if target == "AQI_future_1" else 24
    data = df.dropna(subset=[target]).copy()
    data["_target_time"] = (
        data["_Timestamp"] + pd.to_timedelta(horizon, unit="h")
    )

    train = data[
        data["_year"].between(2020, 2023)
        & (data["_target_time"] < pd.Timestamp("2024-01-01"))
        & data["Station_ID"].isin(tuning_train_station_ids)
    ].copy()

    validation = data[
        (data["_year"] == 2024)
        & (data["_target_time"] < pd.Timestamp("2025-01-01"))
        & data["Station_ID"].isin(tuning_val_station_ids)
    ].copy()

    if train.empty or validation.empty:
        raise ValueError(
            f"{target}: empty tuning split. "
            f"Train={len(train)}, validation={len(validation)}"
        )

    n_val_stations = validation["Station_ID"].nunique()
    if n_val_stations < 2:
        raise ValueError(
            f"{target}: only {n_val_stations} tuning-validation station(s) "
            "have usable 2024 rows. Increase validation fraction or check data."
        )

    print("\n" + "=" * 75)
    print(f"Target: {target}")
    print(f"Train rows: {len(train):,}; validation rows: {len(validation):,}")
    print(f"Validation stations: {n_val_stations}")

    for model_name, columns in model_features.items():
        X_train = clean_X(train, columns)
        y_train = train[target]
        X_val = clean_X(validation, columns)
        y_val = validation[target]

        rows = []

        print(f"\nModel {model_name}: {len(columns)} features")
        for trial, params in enumerate(candidate_params, start=1):
            model = make_tuning_model(params, validation["Station_ID"])
            model.fit(
                X_train,
                y_train,
                eval_set=[(X_val, y_val)],
                verbose=False,
            )

            predictions = predict_gpu_batched(
                model, X_val, best_iteration=model.best_iteration
            )
            macro_mae, station_count = macro_station_mae(
                y_val, predictions, validation["Station_ID"]
            )
            overall_mae = mean_absolute_error(y_val, predictions)
            rmse = float(np.sqrt(mean_squared_error(y_val, predictions)))
            best_trees = int(model.best_iteration) + 1

            rows.append({
                "Target": target,
                "Model": model_name,
                "Trial": trial,
                "Macro_MAE": macro_mae,
                "Overall_MAE": float(overall_mae),
                "RMSE": rmse,
                "R2": float(r2_score(y_val, predictions)),
                "Validation_Stations": station_count,
                "Best_N_Estimators": best_trees,
                "Parameters": json.dumps(params, sort_keys=True),
            })

            print(
                f"Trial {trial:02d}/{len(candidate_params)} | "
                f"macro MAE={macro_mae:.4f} | "
                f"overall MAE={overall_mae:.4f} | trees={best_trees}"
            )

        results = pd.DataFrame(rows).sort_values(
            ["Macro_MAE", "Overall_MAE"], ascending=[True, True]
        ).reset_index(drop=True)

        best = results.iloc[0]
        selected = json.loads(best["Parameters"])
        selected["n_estimators"] = int(best["Best_N_Estimators"])
        PARAMS[target][model_name] = selected

        results.to_csv(
            OUTPUT_DIR / f"tuning_{target}_{model_name}.csv", index=False
        )
        all_results.append(results)
        summary_rows.append({
            "Target": target,
            "Model": model_name,
            "Macro_MAE": float(best["Macro_MAE"]),
            "Overall_MAE": float(best["Overall_MAE"]),
            "RMSE": float(best["RMSE"]),
            "R2": float(best["R2"]),
            "Validation_Stations": int(best["Validation_Stations"]),
            "Parameters": json.dumps(selected, sort_keys=True),
        })

        print(f"Selected parameters for {target}, Model {model_name}:")
        print(json.dumps(selected, indent=2))

# This JSON is loaded by the separate final model/evaluation script.
with open(OUTPUT_DIR / "best_hyperparameters.json", "w", encoding="utf-8") as f:
    json.dump(PARAMS, f, indent=2)

pd.concat(all_results, ignore_index=True).to_csv(
    OUTPUT_DIR / "all_tuning_results.csv", index=False
)
pd.DataFrame(summary_rows).to_csv(
    OUTPUT_DIR / "tuning_summary.csv", index=False
)

print("\nTUNING COMPLETE")
print(f"Parameters saved to: {(OUTPUT_DIR / 'best_hyperparameters.json').resolve()}")
print(json.dumps(PARAMS, indent=2))


Loaded 1,780,876 rows and 134 columns.
Hour values found: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]
Missing Hour values: 0
Project seen stations: 33
Project unseen stations held untouched: 8
Tuning training stations: 26
Tuning validation stations: 7

Target: AQI_future_1
Train rows: 681,855; validation rows: 60,125
Validation stations: 7

Model A: 127 features
Trial 01/30 | macro MAE=1.1890 | overall MAE=1.1973 | trees=1500
Trial 02/30 | macro MAE=1.2727 | overall MAE=1.2821 | trees=882
Trial 03/30 | macro MAE=1.1580 | overall MAE=1.1659 | trees=1500
Trial 04/30 | macro MAE=1.1489 | overall MAE=1.1568 | trees=1500
Trial 05/30 | macro MAE=1.1455 | overall MAE=1.1535 | trees=1491
Trial 06/30 | macro MAE=1.38